In [ ]:
import asyncio
import json
import os
import logging
import time
import csv
from pathlib import Path
from typing import List, Dict, Optional
from urllib.parse import urlparse, parse_qs
import pandas as pd
from firecrawl import FirecrawlApp

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('podcast_scraper.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# Configuration
INPUT_FOLDER = "apollo_podcast_urls"
OUTPUT_CSV = "podcast_scrape_results.csv"
FIRECRAWL_API_KEY = "fc-dd7e9d169ba54032a5c90b28602f45fa"  # Add your Firecrawl API key here
MAX_CONCURRENT_SCRAPES = 50  # Same as map endpoint
BATCH_SIZE = 10  # Write to CSV after processing this many files


In [ ]:
def is_podcast_url(url: str) -> bool:
    """
    Check if a URL is podcast-related using multiple patterns.
    
    Patterns checked:
    1. Contains 'podcast' keyword (case-insensitive)
    2. Is from a known podcast platform
    """
    url_lower = url.lower()
    
    # Pattern 1: Contains 'podcast' keyword
    if 'podcast' in url_lower:
        return True
    
    # Pattern 2: Check if from podcast platform domains
    try:
        parsed = urlparse(url)
        domain = parsed.netloc.lower()
        
        # Remove www. prefix for matching
        domain = domain.replace('www.', '')
        
        # Podcast platform domains
        podcast_domains = [
            'podcasts.apple.com',
            'anchor.fm',
            'buzzsprout.com',
            'podbean.com',
            'libsyn.com',
            'stitcher.com',
        ]
        
        # Check exact domain match
        if domain in podcast_domains:
            return True
        
        # Check for podcast paths in common platforms
        path = parsed.path.lower()
        if domain in ['spotify.com', 'open.spotify.com'] and 'podcast' in path:
            return True
        if domain in ['soundcloud.com'] and 'podcast' in path:
            return True
        if domain in ['iheart.com', 'iheartradio.com'] and 'podcast' in path:
            return True
        if domain in ['audible.com'] and 'podcast' in path:
            return True
        
    except Exception as e:
        logger.warning(f"Error parsing URL {url}: {e}")
        return False
    
    return False


In [ ]:
# URL Ranking Helper Functions

BAD_TOKENS = {"terms", "privacy", "legal", "cookie", "login", "signup", "account", "checkout"}

def norm_domain(url: str) -> str:
    """Normalize domain by removing www. prefix"""
    d = urlparse(url).netloc.lower()
    return d[4:] if d.startswith("www.") else d

def path(url: str) -> str:
    """Extract and normalize path"""
    p = urlparse(url).path or ""
    if not p.startswith("/"):
        p = "/" + p
    return p.rstrip("/") or "/"

def depth(url: str) -> int:
    """Calculate URL depth (number of path segments)"""
    p = path(url)
    if p == "/":
        return 0
    return len([seg for seg in p.strip("/").split("/") if seg])

def is_staging(url: str) -> bool:
    """Detect staging/dev domains"""
    d = norm_domain(url)
    return d.startswith(("stage.", "staging.", "dev.")) or "localhost" in d

def tracking_penalty(url: str) -> int:
    """Detect tracking parameters and return penalty"""
    q = parse_qs(urlparse(url).query)
    keys = {k.lower() for k in q.keys()}
    if any(k.startswith("utm_") for k in keys) or {"gclid", "fbclid"} & keys:
        return 15
    return 0

def score_podcast_url(root_url: str, candidate_url: str) -> int:
    """
    Score a podcast URL candidate. Higher score = better candidate.
    
    Scoring rules:
    - Same domain bonus: +80
    - Exact podcast hub (/podcast or /podcasts): +120
    - Hub paths (/podcast/ or /podcasts/): +90
    - Keyword bonuses: +30 for "podcast" in path, +15 for "episode"/"episodes"/"show"
    - Depth preference: +max(0, 25 - 5*depth)
    - Penalties: -80 for non-content pages, -60 for staging, -15 for tracking params, -10 for fragments
    """
    rd = norm_domain(root_url)
    ud = norm_domain(candidate_url)
    p = path(candidate_url).lower()
    
    score = 0
    
    # Same domain preference
    if ud == rd:
        score += 80
    
    # Prefer podcast hub pages
    if p in ("/podcast", "/podcasts"):
        score += 120
    elif p.startswith("/podcast/") or p.startswith("/podcasts/"):
        score += 90
    
    # Keyword signals
    if "podcast" in p:
        score += 30
    if "episode" in p or "episodes" in p or "show" in p:
        score += 15
    
    # Shallower paths are usually hubs
    d = depth(candidate_url)
    score += max(0, 25 - 5 * d)
    
    # Non-content penalties
    if any(tok in p for tok in BAD_TOKENS):
        score -= 80
    
    # Staging/tracking penalties
    if is_staging(candidate_url):
        score -= 60
    score -= tracking_penalty(candidate_url)
    if "#" in candidate_url:
        score -= 10
    
    return score

def pick_top10(root_url: str, podcast_candidates: List[str]) -> List[str]:
    """
    Rank podcast candidates and select top 10.
    If same-domain hub pages exist, prioritize them first.
    """
    rd = norm_domain(root_url)
    
    # Rank everything
    ranked = sorted(
        set(podcast_candidates),
        key=lambda u: score_podcast_url(root_url, u),
        reverse=True
    )
    
    # If we have same-domain hub pages, force them to the front
    same_domain_hubs = [
        u for u in ranked
        if norm_domain(u) == rd and path(u).lower() in ("/podcast", "/podcasts")
    ]
    if same_domain_hubs:
        rest = [u for u in ranked if u not in same_domain_hubs]
        return (same_domain_hubs + rest)[:10]
    
    return ranked[:10]


In [ ]:
def scrape_url_with_firecrawl_sync(url: str, app: FirecrawlApp) -> Dict:
    """
    Synchronous function to scrape a URL using Firecrawl Python SDK.
    
    Returns:
        dict with 'success' (bool) and either 'markdown' (str) or 'error' (str)
    """
    try:
        # Pass parameters directly as keyword arguments (v2 API expects top-level params, not nested)
        result = app.scrape_url(
            url,
            formats=["markdown"],
            onlyMainContent=False,
            maxAge=172800000
        )
        
        if getattr(result, 'success', False):
            # Extract markdown content from result
            # Try different possible attributes based on SDK response structure
            markdown = ""
            if hasattr(result, 'markdown'):
                markdown = result.markdown
            elif hasattr(result, 'data'):
                data = result.data
                if isinstance(data, dict):
                    markdown = data.get('markdown', '')
                elif hasattr(data, 'markdown'):
                    markdown = data.markdown
            
            return {"success": True, "markdown": markdown}
        else:
            error_msg = getattr(result, 'error', 'Unknown error') or str(result)
            return {"success": False, "error": error_msg}
            
    except Exception as e:
        error_str = str(e)
        # Check for rate limit error
        if "rate limit" in error_str.lower() or "429" in error_str:
            return {"success": False, "error": error_str}
        return {"success": False, "error": str(e)}


In [ ]:
async def scrape_podcast_url(
    url: str,
    root_url: str,
    app: FirecrawlApp,
    semaphore: asyncio.Semaphore,
    stats: Dict
) -> Dict:
    """
    Async function to scrape single podcast URL with semaphore control.
    Uses run_in_executor to run synchronous Firecrawl SDK call.
    
    Returns:
        dict with 'root_url', 'podcast_url', and 'scraped_information'
    """
    async with semaphore:
        try:
            stats['total_scrapes'] += 1
            # Run synchronous Firecrawl SDK call in executor
            loop = asyncio.get_event_loop()
            result = await loop.run_in_executor(None, scrape_url_with_firecrawl_sync, url, app)
            
            if result['success']:
                stats['scrapes_success'] += 1
                scraped_info = result['markdown']
            else:
                stats['scrapes_failed'] += 1
                scraped_info = result['error']
            
            return {
                "root_url": root_url,
                "podcast_url": url,
                "scraped_information": scraped_info
            }
        except Exception as e:
            stats['scrapes_failed'] += 1
            logger.error(f"Error scraping {url}: {e}")
            return {
                "root_url": root_url,
                "podcast_url": url,
                "scraped_information": f"Error: {str(e)}"
            }

async def process_json_file(
    file_path: Path,
    app: FirecrawlApp,
    semaphore: asyncio.Semaphore,
    stats: Dict
) -> List[Dict]:
    """
    Process a single JSON file: load, filter, rank, and scrape top 10 podcast URLs.
    
    Returns:
        List of row dicts with root_url, podcast_url, scraped_information
    """
    try:
        # Load JSON array from file (synchronous file I/O)
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        stats['files_processed'] += 1
        
        # Validate it's a list
        if not isinstance(data, list) or len(data) == 0:
            logger.warning(f"{file_path.name}: Empty or invalid JSON array")
            return []
        
        # Extract root_url from data[0] (following extract_top_urls.py logic)
        root_url = data[0]
        if not isinstance(root_url, str):
            logger.warning(f"{file_path.name}: First element is not a URL string")
            return []
        
        # Filter podcast URLs
        podcast_candidates = [
            u for u in data
            if isinstance(u, str) and is_podcast_url(u)
        ]
        
        if not podcast_candidates:
            logger.debug(f"{file_path.name}: No podcast URLs found")
            return []
        
        stats['files_with_podcasts'] += 1
        logger.info(f"{file_path.name}: Found {len(podcast_candidates)} podcast URLs, selecting top 10")
        
        # Rank and select top 10
        top10 = pick_top10(root_url, podcast_candidates)
        
        if not top10:
            return []
        
        # Create async tasks for all top10 URLs
        tasks = [
            scrape_podcast_url(url, root_url, app, semaphore, stats)
            for url in top10
        ]
        
        # Scrape all URLs concurrently
        results = await asyncio.gather(*tasks, return_exceptions=True)
        
        # Filter out exceptions
        valid_results = []
        for result in results:
            if isinstance(result, Exception):
                logger.error(f"Exception in scrape task: {result}")
                stats['errors'] += 1
            elif isinstance(result, dict):
                valid_results.append(result)
        
        return valid_results
        
    except json.JSONDecodeError as e:
        logger.error(f"{file_path.name}: JSON decode error: {e}")
        stats['errors'] += 1
        return []
    except Exception as e:
        logger.error(f"{file_path.name}: Error processing file: {e}")
        stats['errors'] += 1
        return []


In [ ]:
def write_rows_to_csv(rows: List[Dict], output_path: Path, write_header: bool = False):
    """
    Write rows to CSV file. If file doesn't exist, write header.
    """
    file_exists = output_path.exists()
    
    with open(output_path, 'a', newline='', encoding='utf-8') as f:
        fieldnames = ['root_url', 'podcast_url', 'scraped_information']
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        
        # Write header if file is new or if explicitly requested
        if not file_exists or write_header:
            writer.writeheader()
        
        # Write rows
        for row in rows:
            writer.writerow(row)

async def process_all_files(api_key: str) -> int:
    """
    Main async function to process all JSON files concurrently.
    Writes results to CSV in batches as files are processed.
    
    Returns:
        Total number of rows written to CSV
    """
    logger.info("=" * 60)
    logger.info("Starting Podcast URL Scraper")
    logger.info(f"Input folder: {INPUT_FOLDER}")
    logger.info(f"Output CSV: {OUTPUT_CSV}")
    logger.info(f"Max concurrent scrapes: {MAX_CONCURRENT_SCRAPES}")
    logger.info(f"Batch size: {BATCH_SIZE} files per CSV write")
    logger.info("=" * 60)
    
    # Get all JSON files from INPUT_FOLDER
    input_path = Path(INPUT_FOLDER)
    if not input_path.exists():
        logger.error(f"Input folder does not exist: {INPUT_FOLDER}")
        return 0
    
    json_files = list(input_path.glob("*.json"))
    total_files = len(json_files)
    
    if total_files == 0:
        logger.warning(f"No JSON files found in {INPUT_FOLDER}")
        return 0
    
    logger.info(f"Found {total_files} JSON files to process")
    
    # Initialize CSV file (create/truncate and write header)
    output_path = Path(OUTPUT_CSV)
    if output_path.exists():
        output_path.unlink()  # Remove existing file to start fresh
    write_rows_to_csv([], output_path, write_header=True)
    logger.info(f"Initialized CSV file: {OUTPUT_CSV}")
    
    # Initialize FirecrawlApp
    app = FirecrawlApp(api_key=api_key)
    
    # Initialize statistics dict
    stats = {
        'files_processed': 0,
        'files_with_podcasts': 0,
        'total_scrapes': 0,
        'scrapes_success': 0,
        'scrapes_failed': 0,
        'errors': 0,
        'rows_written': 0
    }
    
    # Create semaphore for rate limiting
    semaphore = asyncio.Semaphore(MAX_CONCURRENT_SCRAPES)
    
    # Create async tasks for all JSON files
    tasks = {
        asyncio.create_task(process_json_file(file_path, app, semaphore, stats)): file_path
        for file_path in json_files
    }
    
    logger.info(f"Processing {len(tasks)} files with up to {MAX_CONCURRENT_SCRAPES} concurrent scrapes...")
    start_time = time.time()
    
    # Process files as they complete and write in batches
    batch_rows = []
    completed_files = 0
    
    for task in asyncio.as_completed(tasks.keys()):
        try:
            result = await task
            completed_files += 1
            
            if isinstance(result, Exception):
                logger.error(f"Exception in file processing: {result}")
                stats['errors'] += 1
            elif isinstance(result, list) and result:
                batch_rows.extend(result)
                stats['rows_written'] += len(result)
            
            # Write to CSV in batches (every BATCH_SIZE files)
            if completed_files % BATCH_SIZE == 0 and batch_rows:
                write_rows_to_csv(batch_rows, output_path)
                logger.info(f"📝 Wrote batch: {len(batch_rows)} rows to CSV (Total: {stats['rows_written']} rows, Files: {completed_files}/{total_files})")
                batch_rows = []
            
            # Progress update every 10 files
            if completed_files % 10 == 0:
                elapsed = time.time() - start_time
                rate = completed_files / elapsed if elapsed > 0 else 0
                eta = (total_files - completed_files) / rate if rate > 0 else 0
                logger.info(f"Progress: {completed_files}/{total_files} files ({completed_files/total_files*100:.1f}%) | "
                          f"Rate: {rate:.2f} files/sec | ETA: {eta/60:.1f} min | "
                          f"Rows written: {stats['rows_written']}")
                
        except Exception as e:
            logger.error(f"Error processing completed task: {e}")
            stats['errors'] += 1
    
    # Write any remaining rows
    if batch_rows:
        write_rows_to_csv(batch_rows, output_path)
        logger.info(f"📝 Wrote final batch: {len(batch_rows)} rows to CSV")
    
    elapsed_time = time.time() - start_time
    
    # Log final statistics
    logger.info("=" * 60)
    logger.info("Processing Complete!")
    logger.info(f"Total files processed: {stats['files_processed']}")
    logger.info(f"Files with podcast URLs: {stats['files_with_podcasts']}")
    logger.info(f"Total scrapes attempted: {stats['total_scrapes']}")
    logger.info(f"Successful scrapes: {stats['scrapes_success']}")
    logger.info(f"Failed scrapes: {stats['scrapes_failed']}")
    logger.info(f"Errors encountered: {stats['errors']}")
    logger.info(f"Total rows written to CSV: {stats['rows_written']}")
    logger.info(f"Time elapsed: {elapsed_time:.2f} seconds")
    logger.info(f"CSV file: {output_path.absolute()}")
    logger.info("=" * 60)
    
    return stats['rows_written']


In [ ]:
# Main execution
if not FIRECRAWL_API_KEY:
    logger.error("FIRECRAWL_API_KEY is not set. Please add your API key in the configuration cell.")
else:
    # Process all files (writes to CSV incrementally)
    total_rows = await process_all_files(FIRECRAWL_API_KEY)
    
    if total_rows > 0:
        # Read the CSV to display summary
        output_path = Path(OUTPUT_CSV)
        df = pd.read_csv(output_path)
        
        logger.info(f"CSV file contains {len(df)} rows")
        
        # Display summary
        print(f"\n{'='*60}")
        print(f"Final Summary:")
        print(f"  Total rows in CSV: {len(df)}")
        print(f"  Unique root URLs: {df['root_url'].nunique()}")
        print(f"  Unique podcast URLs: {df['podcast_url'].nunique()}")
        print(f"  Successful scrapes: {(~df['scraped_information'].str.startswith('Error:', na=False)).sum()}")
        print(f"  Failed scrapes: {df['scraped_information'].str.startswith('Error:', na=False).sum()}")
        print(f"  Output file: {output_path.absolute()}")
        print(f"{'='*60}\n")
        
        # Display first few rows
        print("First 5 rows:")
        print(df.head())
    else:
        logger.warning("No results were written to CSV. Check logs for details.")
